## ENEM 2023 — Pré-processamento (microdados)

## Objetivo
Consolidar variáveis socioeconômicas e notas em um dataset padronizado (nível candidato), com tipagem consistente e categorias ordenadas.

---

## Estratégia de processamento

O pipeline foi estruturado em duas etapas complementares:

### 1) Ingestão e padronização inicial (DuckDB + SQL)
- Leitura eficiente dos arquivos CSV originais
- Seleção de colunas relevantes
- Padronização de nomes
- Exportação para formato Parquet

Essa etapa foi implementada fora do notebook, via scripts SQL e DuckDB, com foco em desempenho e reprodutibilidade.

### 2) Pré-processamento analítico (Pandas)
- Tratamento de valores ausentes
- Recodificação de variáveis socioeconômicas
- Tipagem e ordenação categórica
- Preparação para análise e modelagem

Este notebook cobre **exclusivamente a segunda etapa**, assumindo como entrada os dados já estruturados em Parquet.

---

## Entradas
- Arquivo Parquet derivados dos microdados ENEM 2023

## Saídas
- `DADOS_TRATADOS_2023.parquet`


## 📌 Observações sobre os dados (ENEM 2024)

Em 2024, o INEP disponibilizou os microdados em dois arquivos separados:

- **Participantes (dados socioeconômicos)**
- **Resultados (notas)**

Além disso:

- Os identificadores foram anonimizados
- Não é possível relacionar diretamente os dois arquivos

Fonte oficial:  
https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/microdados/enem

---

## 📌 Observação sobre o repositório

Os microdados brutos não são versionados neste repositório devido a:

- tamanho elevado dos arquivos
- restrições de redistribuição

Consulte o `README.md` para instruções de obtenção.

### 1) Ambiente e imports 

In [1]:
import sys
from pathlib import Path

# Permite importar o pacote `src/` a partir do diretório do projeto.
ROOT_PATH = Path().resolve().parents[1]  # notebooks/00_preprocessamento -> projeto
if str(ROOT_PATH) not in sys.path:
    sys.path.append(str(ROOT_PATH))

import re
import numpy as np
import pandas as pd

import seaborn as sns


from src.config import (
    BASE_BRUTA_2023,
    DADOS_TRATADOS_2023,
)    


from src.preprocessamento.agregacoes import recodificar_microdados_enem
from src.preprocessamento.categorias import (
    ORDEM_ESCOLA, 
    ORDEM_ESTADO_CIVIL,
    ORDEM_FAIXA_ETARIA, 
    ORDEM_LINGUA,    
    ORDEM_OCUPACAO,
    ORDEM_PAIS_ESCOLARIDADE, 
    ORDEM_RACA,
    ORDEM_SAL_MIN, 
    ORDEM_SEXO,


)


pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")


### 2) Carregamento dos dados tratados

Os dados são carregados a partir dos arquivos Parquet gerados na etapa de ingestão.

Essa abordagem reduz significativamente o consumo de memória e o tempo de processamento.

### Arquitetura do pipeline

O projeto adota uma arquitetura híbrida:

- **DuckDB + SQL** → ingestão e padronização (dados brutos → Parquet)
- **Pandas** → transformação analítica e modelagem

Essa separação permite:

- melhor desempenho no processamento de grandes volumes
- maior organização do código
- reutilização das bases tratadas em múltiplas análises

In [2]:
df = pd.read_parquet(BASE_BRUTA_2023)

df.head(3)

,faixa_etaria,sexo,estado_civil,cor_raca,escola,municipio,uf,nota_cn,nota_ch,nota_lc,nota_mt,lingua,nota_redacao,escolaridade_pai,escolaridade_mae,ocup_pai,ocup_mae,n_pessoas_resd,sal_min,emp_domst,banh,quartos,car,moto,gelad,freezer,lv_rp,sc_rp,mcr_ond,lv_lc,asp_po,tv,dvd,tv_assin,cel,tel_fix,comptdr,internet
0,14,M,2,1,1,Brasília,DF,None,None,None,None,0,None,A,F,E,D,5,F,C,C,D,C,D,C,B,B,D,C,C,B,B,A,B,B,A,A,B
1,12,M,2,1,1,Brasília,DF,None,None,None,None,0,None,F,E,E,B,3,H,A,B,C,C,A,B,B,B,A,B,A,B,B,A,A,C,A,D,B
2,6,F,1,1,1,Caxias do Sul,RS,502,498.9,475.6,363.2,1,700,H,E,C,F,5,C,A,B,D,B,A,B,A,B,A,B,A,A,B,A,A,A,A,A,B


### 3) Tratamento de valores ausentes
Valores ausentes

Ausências em notas podem refletir não comparecimento/condições específicas — mantidas para decisões posteriores (depende do objetivo).

Ausências em renda_mens são raras e removidas por inviabilizarem a classificação socioeconômica.

Para variáveis categóricas socioeconômicas a exclusão de registros com NaN representaria uma perda significativa de 
informação e potencialmente introduziria viés na amostra.


#### Categoria Não informada(o)

1. **Reconhecimento** que a não-resposta é um dado legítimo
2. **Preservação** a distinção entre diferentes tipos de missing values
3. **Manutenção** a integridade da estrutura categórica ordinal/nominal

Os microdados do ENEM já preveem códigos específicos para "não respondido" ou "não se aplica" em diversas variáveis:

- **Cor/raça:** código 0 ou 6 = "não declarada"
- **Estado civil:** código 0 = "não informado"
- **Escolaridade dos pais:** categoria H = "desconhecido"

Nossa abordagem estende essa lógica de forma consistente para todas as variáveis categóricas, 
incluindo aquelas onde o INEP não prevê explicitamente um código para não-resposta.

In [3]:
df.isna().sum()

faixa_etaria              0
sexo                      0
estado_civil              0
cor_raca                  0
escola                    0
municipio                 0
uf                        0
nota_cn             1241528
nota_ch             1111312
nota_lc             1111312
nota_mt             1241528
lingua                    0
nota_redacao        1111312
escolaridade_pai          0
escolaridade_mae          0
ocup_pai                  0
ocup_mae                  0
n_pessoas_resd            0
sal_min                   0
emp_domst                 0
banh                      0
quartos                   0
car                       0
moto                      0
gelad                     0
freezer                   0
lv_rp                     0
sc_rp                     0
mcr_ond                   0
lv_lc                     0
asp_po                    0
tv                        0
dvd                       0
tv_assin                  0
cel                       0
tel_fix             

### 4) Recodificações, Tipagem e Ordenação
Recodificações socioeconômicas

As respostas do questionário são recodificadas para categorias interpretáveis e/ou variáveis numéricas, mantendo rastreabilidade e consistência longitudinal (2021–2024 quando aplicável).

In [4]:
colunas_abcde = ["banh","quartos","car","moto","gelad","freezer","lv_rp","sc_rp","tv","mcr_ond","lv_lc","cel","comptdr"]
colunas_ab = ["dvd","asp_po","tv_assin","tel_fix","internet"]

df = recodificar_microdados_enem(
    df,
    schema_escola="enem_2021_2023",
    col_bens_abcde=colunas_abcde,
    col_bens_ab=colunas_ab,
)

# ordenar categorias
df["cor_raca"] = df["cor_raca"].astype("category").cat.set_categories(ORDEM_RACA, ordered=True)
df["escola"] = df["escola"].astype("category").cat.set_categories(ORDEM_ESCOLA, ordered=True)
df["escolaridade_pai"] = df["escolaridade_pai"].astype("category").cat.set_categories(ORDEM_PAIS_ESCOLARIDADE, ordered=True)
df["escolaridade_mae"] = df["escolaridade_mae"].astype("category").cat.set_categories(ORDEM_PAIS_ESCOLARIDADE, ordered=True)
df["estado_civil"] = df["estado_civil"].astype("category").cat.set_categories(ORDEM_ESTADO_CIVIL, ordered=True)
df["faixa_etaria"] = df["faixa_etaria"].astype("category").cat.set_categories(ORDEM_FAIXA_ETARIA, ordered=True)
df["lingua"] = df["lingua"].astype("category").cat.set_categories(ORDEM_LINGUA, ordered=True)
df["ocup_pai"] = df["ocup_pai"].astype("category").cat.set_categories(ORDEM_OCUPACAO, ordered=True)
df["ocup_mae"] = df["ocup_mae"].astype("category").cat.set_categories(ORDEM_OCUPACAO, ordered=True)
df["sal_min"] = df["sal_min"].astype("category").cat.set_categories(ORDEM_SAL_MIN, ordered=True)
df["sexo"] = df["sexo"].astype("category").cat.set_categories(ORDEM_SEXO, ordered=True)


# municipio e uf: são categóricas, mas NÃO precisam de ordem
df["municipio"] = df["municipio"].astype("category")
df["uf"] = df["uf"].astype("category")

# colunas float
for coluna in df.filter(like="nota").columns:
    df[coluna] = pd.to_numeric(df[coluna], errors="coerce").astype("float32")


# colunas int
COLUNAS_INT=["n_pessoas_resd","emp_domst"]+colunas_abcde + colunas_ab

for coluna in COLUNAS_INT:
    df[coluna] = pd.to_numeric(df[coluna], errors="coerce").astype("Int8")


### 5) Criação de Índice de Consumo
Criar um índice sintético com todas as colunas de consumo


In [5]:
CONSUMO_COLS = colunas_abcde + colunas_ab

consumo = df[CONSUMO_COLS].copy()

# substitui NaN por 0 (se a codificação for consistente)
consumo = consumo.fillna(0)

# normaliza cada coluna para 0-1 usando o máximo observado
consumo_norm = consumo / (consumo.max() + 1e-9)

df["indice_consumo"] = consumo_norm.mean(axis=1).astype("float32")
df["indice_consumo"] = df["indice_consumo"].round(4)

### 6) Dropar colunas de consumo não essenciais

In [6]:
ESSENCIAIS = ["gelad", "lv_rp", "comptdr", "cel", "tv", "indice_consumo"]

# garante que as essenciais existem
missing = [c for c in ESSENCIAIS if c not in df.columns]
if missing:
    raise ValueError(f"Colunas essenciais ausentes: {missing}")

# remove as demais colunas de consumo, mas mantém as essenciais e o índice
cols_consumo_drop = [c for c in CONSUMO_COLS if c not in ESSENCIAIS]
df = df.drop(columns=cols_consumo_drop)


In [7]:
df.head(3)

,faixa_etaria,sexo,estado_civil,cor_raca,escola,municipio,uf,nota_cn,nota_ch,nota_lc,nota_mt,lingua,nota_redacao,escolaridade_pai,escolaridade_mae,ocup_pai,ocup_mae,n_pessoas_resd,sal_min,emp_domst,gelad,lv_rp,tv,cel,comptdr,indice_consumo
0,36-45,masculino,casado/mora com companheiro,branca,não informada,Brasília,DF,NaN,NaN,NaN,NaN,inglês,NaN,não estudou,superior,superior,médio/tec,5,1 a 3,3,2,1,1,1,0,0.49
1,26-35,masculino,casado/mora com companheiro,branca,não informada,Brasília,DF,NaN,NaN,NaN,NaN,inglês,NaN,superior,médio,superior,básico,3,3 a 5,0,1,1,1,2,3,0.32
2,20-25,feminino,solteiro,branca,não informada,Caxias do Sul,RS,502.00,498.90,475.60,363.20,espanhol,700.00,desconhecida,médio,manual/tec,desconhecida,5,1 a 3,0,1,1,1,0,0,0.18


### 7) Conferências finais e exportação
Exportação (Parquet)

O dataset tratado é persistido em Parquet por eficiência (compressão + leitura rápida).

In [8]:
df.to_parquet(DADOS_TRATADOS_2023, index=False)

print("✅ Salvo em:", DADOS_TRATADOS_2023)
print("shape:", df.shape)
print("memória (MB):", df.memory_usage(deep=True).sum() / 1_048_576)

✅ Salvo em: E:\ciencias_dados\projetos\projeto_enem_ml\dados\intermediarios\dados_tratados_2023.parquet
shape: (3933955, 26)
memória (MB): 195.29591846466064
